# Lesson 2.1 — The environment and the rollout loop

This notebook builds the intuition the GPT/agent-loop comparison depends on: the
Gymnasium call pattern **is** the embodied agent loop.

```text
o_t  ->  policy  ->  a_t  ->  env.step  ->  o_{t+1}, r, terminated, truncated
```

Adapted from the archived probes
`archive/lesson_0_1/smoke_test_maniskill.py` and
`archive/lesson_0_1/collect_pickcube_random.py`, which established this pattern
for Lessons 0–1 and are now frozen.

## 2.1.1 — Create and reset

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env


env = gym.make("PickCube-v1", obs_mode="state", control_mode="pd_joint_delta_pos", num_envs=1)

print("environment :", type(env.unwrapped).__name__)
print("obs space   :", env.observation_space)
print("action space:", env.action_space)
print("control freq:", env.unwrapped.control_freq, "Hz")

obs, info = env.reset(seed=0)
print("\nreset -> obs shape:", obs.shape, "dtype:", obs.dtype)
print("info keys:", list(info.keys()))

2026-09-22 11:14:22,321 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


environment : PickCubeEnv
obs space   : Box(-inf, inf, (1, 42), float32)
action space: Box(-1.0, 1.0, (8,), float32)
control freq: 20 Hz

reset -> obs shape: torch.Size([1, 42]) dtype: torch.float32
info keys: ['elapsed_steps', 'success', 'is_obj_placed', 'is_robot_static', 'is_grasped', 'reconfigure']


## 2.1.2 — One step: the five return values

`env.step(action)` returns `(observation, reward, terminated, truncated, info)`.
Two separate stop signals is a common source of bugs:

- `terminated` — the task ended in success or unrecoverable failure;
- `truncated` — the episode hit a step or time limit.

An episode that is `truncated` tells you nothing about success. The project's
source trajectory ends exactly this way, which is why
`notes/progress.md` records `success_any=False`.

In [2]:
action = env.action_space.sample()
print("action:", action, "shape:", action.shape)

next_obs, reward, terminated, truncated, info = env.step(action)

print("\nnext_obs shape:", next_obs.shape)
print("reward        :", reward)
print("terminated    :", terminated)
print("truncated     :", truncated)
print("info          :", info)

action: [ 0.62323415 -0.5443854   0.82626545  0.22426002 -0.08090118 -0.9197369
 -0.27873144  0.39224908] shape: (8,)

next_obs shape: torch.Size([1, 42])
reward        : tensor([0.0617])
terminated    : tensor([False])
truncated     : tensor([False])
info          : {'elapsed_steps': tensor([1], dtype=torch.int32), 'success': tensor([False]), 'is_obj_placed': tensor([False]), 'is_robot_static': tensor([False]), 'is_grasped': tensor([False])}


## 2.1.3 — A full random episode

A rollout is just this loop. Random actions are used on purpose: the goal here is
to validate the mechanics, not to solve the task.

Note the T-versus-state bookkeeping: the loop performs T steps and stores **T**
observations, actions, and rewards. The observation *after* the last action is not
stored unless you explicitly keep it. That off-by-one is the subject of
`2.3_time_alignment.ipynb`.

In [3]:
def collect_random_episode(env, seed=0, max_steps=50, verbose=False):
    """Run a random rollout and record the transitions.

    Returns T observations, T actions, T rewards where T = max_steps.
    """
    obs, info = env.reset(seed=seed)
    observations, actions, rewards = [], [], []
    terminated_flags, truncated_flags = [], []

    for step in range(max_steps):
        action = env.action_space.sample()
        next_obs, reward, terminated, truncated, info = env.step(action)

        observations.append(obs[0].cpu().numpy())
        actions.append(action.copy())
        rewards.append(float(reward.item()))
        terminated_flags.append(bool(terminated.item()))
        truncated_flags.append(bool(truncated.item()))

        if verbose and step < 5:
            print(f"  step {step:2d}: reward={float(reward.item()):+.4f} "
                  f"terminated={bool(terminated.item())} truncated={bool(truncated.item())}")

        obs = next_obs
        if bool(terminated.item()) or bool(truncated.item()):
            break

    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "terminated": np.asarray(terminated_flags),
        "truncated": np.asarray(truncated_flags),
        "info": info,
    }


episode = collect_random_episode(env, seed=0, max_steps=50, verbose=True)

print("\nobservations:", episode["observations"].shape)
print("actions     :", episode["actions"].shape)
print("rewards     :", episode["rewards"].shape)
print("total reward:", episode["rewards"].sum())
print("any success :", "success" in episode["info"])
print("terminated  :", episode["terminated"].any(), "| truncated:", episode["truncated"].any())

  step  0: reward=+0.0560 terminated=False truncated=False
  step  1: reward=+0.0495 terminated=False truncated=False
  step  2: reward=+0.0522 terminated=False truncated=False
  step  3: reward=+0.0511 terminated=False truncated=False
  step  4: reward=+0.0429 terminated=False truncated=False

observations: (50, 42)
actions     : (50, 8)
rewards     : (50,)
total reward: 2.1162405
any success : True
terminated  : False | truncated: True


## 2.1.4 — What "success" actually means here

`PickCube-v1` does not treat "the gripper is holding the cube" as success. Its
evaluation requires the cube to be **placed within the goal threshold** *and* the
robot to be **static**. Grasping alone is not success.

This matters for Lesson 2.9 and beyond: a demonstration is not expert data because
the arm moved plausibly. There must be an explicit success signal.

In [4]:
env.reset(seed=0)
print("goal threshold   :", env.unwrapped.goal_thresh)
print("cube half size   :", env.unwrapped.cube_half_size)
print("cube position    :", env.unwrapped.cube.pose.p[0])
print("goal position    :", env.unwrapped.goal_site.pose.p[0])

distance = (env.unwrapped.cube.pose.p[0] - env.unwrapped.goal_site.pose.p[0]).norm()
print(f"current cube-to-goal distance: {distance.item():.4f} m")
print("episode starts far from the goal, and random actions do not close that gap.")

goal threshold   : 0.025
cube half size   : 0.02
cube position    : tensor([-0.0007,  0.0536,  0.0200])
goal position    : tensor([ 0.0268, -0.0020,  0.2889])
current cube-to-goal distance: 0.2760 m
episode starts far from the goal, and random actions do not close that gap.


## Takeaways

1. `reset` returns `(obs, info)`; `step` returns five values, with `terminated` and
   `truncated` as **separate** stop conditions.
2. A rollout loop is the embodied agent loop. The policy is the only replaceable
   part.
3. T steps produce T observations, T actions, T rewards.
4. In `PickCube-v1`, success requires placement within the goal threshold plus a
   static robot — grasping alone is insufficient. Random rollouts are pipeline
   fixtures, never expert demonstrations.

Next: the dataset layer, starting at `2.3_time_alignment.ipynb`.